In [ ]:
from langgraph_sdk import get_client

client = get_client(url="http://localhost:2024")

assistant_id = "efbf07f8-c28f-4db8-a7ff-17b58c4af012"

# 删除所有线程，免得看晕了
threads = await client.threads.search(limit=100)
for t in threads:
    await client.threads.delete(thread_id=t['thread_id'])

thread = await client.threads.create(
    metadata={"__name__": "多任务策略"}
)
thread_id = thread["thread_id"]
thread_id

# 多任务策略

多任务策略是 Agent Server 独有的功能。

它是指同一个线程内，前一个任务(run)还在运行，此时又发起一个新任务(run)，该如何处理？

比如`Run A`正在运行，此时发起`Run B`，如何处理`A`和`B`。

Agent Server 提供四种处理策略：

| 策略           | Run A        | Run B               | Run A 已产生的状态 |
| -------------- | ------------ | ------------------- | ------------------ |
| `enqueue` 默认 | 继续完成     | 排队等待            | 保留               |
| `reject`       | 继续完成     | 拒绝创建            | 保留               |
| `interrupt`    | 被中断       | 立即接替执行        | 保留已提交部分     |
| `rollback`     | 被中断并删除 | 从 A 之前的状态执行 | 回滚并删除         |

In [ ]:
import asyncio

run1 = await client.runs.create(
    thread_id=thread_id,
    assistant_id=assistant_id,
    input={
        "messages": [
            {"role":"user", "content":"你好！"}
        ]
    },
)

# 等任务1运行一秒钟，模拟两次任务的时间差
await asyncio.sleep(1)

run2 = await client.runs.create(
    thread_id=thread_id,
    assistant_id=assistant_id,
    input={
        "messages": [
            {"role":"user", "content":"hello！"}
        ]
    },
    multitask_strategy="rollback"
)

In [ ]:
current_run1 = await client.runs.get(thread_id, run1["run_id"])
current_run2 = await client.runs.get(thread_id, run2["run_id"])

current_run1['status'], current_run2['status']